# LLaMA LoRA Fine-Tuning for Connections

Fine-tunes LLaMA 3.1 with LoRA on the Connections training split, then evaluates
on the test split using the same metrics as the other notebooks.

**Prerequisites:**
- `pip install transformers peft torch datasets huggingface_hub`
- Set the `HF_API_KEY` environment variable with your Hugging Face token
  (needs access to `meta-llama/Llama-3.1-8B-Instruct`).

In [ ]:
import os
from huggingface_hub import login

login(token=os.environ["HF_API_KEY"])

In [ ]:
from data_loader import get_train_test_split

ds_train, ds_test = get_train_test_split()
print("Training puzzles:", len(ds_train))
print("Testing puzzles:", len(ds_test))

## 1. Fine-tune LLaMA with LoRA

In [ ]:
from conn.llama_fine_tuning import finetune_llama_lora

model, tokenizer, stats = finetune_llama_lora(
    puzzles=list(ds_train),
    epochs=3,
    batch_size=2,
    learning_rate=2e-4,
    lora_r=8,
    lora_alpha=16,
    adapter_output_dir="./adapters/llama-lora-connections",
    verbose=True,
)

print(f"\nTraining complete — {stats.steps} steps")
print("Epoch losses:", [f"{l:.4f}" for l in stats.average_losses])

## 2. (Optional) Load a previously saved adapter

Skip this cell if you just trained above.

In [ ]:
# from conn.llama_fine_tuning import load_llama_lora
#
# model, tokenizer = load_llama_lora("./adapters/llama-lora-connections")

## 3. Evaluate fine-tuned model (zero-shot)

In [ ]:
from conn.solvers.llama import LlamaSolver
from conn import accuracy_min_swaps, accuracy_zero_one, evaluate

solver_ft_zero = LlamaSolver(
    model=model,
    tokenizer=tokenizer,
    k=0,
)

results_zero = evaluate(
    ds_test,
    metric_fns={"zero_one": accuracy_zero_one, "min_swaps": accuracy_min_swaps},
    solver_fn=solver_ft_zero.solve,
    verbose=True,
)

acc, n_eval = results_zero["zero_one"]
mean_swaps, _ = results_zero["min_swaps"]
print(f"\nFine-tuned k=0 | Zero-one accuracy: {acc:.4f}  (n={n_eval})")
print(f"Fine-tuned k=0 | Mean min swaps: {mean_swaps:.2f}")

## 4. Evaluate fine-tuned model (few-shot, k=3)

In [ ]:
solver_ft_few = LlamaSolver(
    model=model,
    tokenizer=tokenizer,
    example_source=ds_train,
    k=3,
)

results_few = evaluate(
    ds_test,
    metric_fns={"zero_one": accuracy_zero_one, "min_swaps": accuracy_min_swaps},
    solver_fn=solver_ft_few.solve,
    verbose=True,
)

acc, n_eval = results_few["zero_one"]
mean_swaps, _ = results_few["min_swaps"]
print(f"\nFine-tuned k=3 | Zero-one accuracy: {acc:.4f}  (n={n_eval})")
print(f"Fine-tuned k=3 | Mean min swaps: {mean_swaps:.2f}")

## 5. Comparison: Base model (no fine-tuning)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

BASE_MODEL = "meta-llama/Llama-3.1-8B-Instruct"

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

base_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16 if device.type == "cuda" else torch.float32,
).to(device)
base_model.eval()

solver_base = LlamaSolver(
    model=base_model,
    tokenizer=base_tokenizer,
    example_source=ds_train,
    k=3,
)

results_base = evaluate(
    ds_test,
    metric_fns={"zero_one": accuracy_zero_one, "min_swaps": accuracy_min_swaps},
    solver_fn=solver_base.solve,
    verbose=True,
)

acc, n_eval = results_base["zero_one"]
mean_swaps, _ = results_base["min_swaps"]
print(f"\nBase k=3 | Zero-one accuracy: {acc:.4f}  (n={n_eval})")
print(f"Base k=3 | Mean min swaps: {mean_swaps:.2f}")